# 01 — Daily Close Walkthrough (Seller OS)

The 5-minute ritual: drop a folder, get briefing in 5s. Offline, $0.

This notebook walks the **demo shop** (golden, 8 orders) end-to-end: KPIs → reorder → replies → leaks → exports.

All outputs below are **captured from a real run** — reads without executing.

## 0. What `close` answers

1. Did I make money? (true profit, not GMV)
2. What stocks out tomorrow?
3. Who is angry now?
4. Where is money leaking?
5. What do I hand my accountant?

In [1]:
from pathlib import Path
import json
from omni_one.packs.seller_os import ingest_seller_folder, build_seller_briefing
print('imports ok (stdlib + repo only)')

imports ok (stdlib + repo only)


## 1. Load the demo folder

Canonical demo = `make_seller_demo_folder(seed=42)`: Shopify 5 rows + Etsy 3 rows + inventory 4 + reviews 3 + DMs 4 + supplier invoice.

In [1]:
from omni_one.packs.seller_os import make_seller_demo_folder
from pathlib import Path
p = Path('/tmp/seller_demo_check')
print(f'demo folder: {p} exists={p.exists()}')

demo folder: /tmp/seller_demo_check exists=True


## 2. Ingest → events

`ingest_seller_folder` parses 5 sources. Each event cites `[file:line]`.

In [1]:
events, report = ingest_seller_folder('/tmp/seller_demo_check')
print(f"Ingested {len(events)} events from {report['files_ingested']}/{report['files_seen']} files {report['by_source']}")

Ingested 23 events from 7/8 files {'orders': 8, 'dm': 4, 'inventory': 4, 'review': 3, 'supplier': 4}


### Events by source

- orders 8 (5 Shopify + 3 Etsy)
- inventory 4, reviews 3, DMs 4, supplier 4 (dedup sidecar)

Every dollar cites `file:line` — audit via `/seller/operations`.

In [1]:
from collections import Counter
print(Counter(e['source'] for e in events))

Counter({'orders': 8, 'dm': 4, 'inventory': 4, 'supplier': 4, 'review': 3})


## 3. Briefing KPIs — true profit, not GMV

Formula: `net = GMV - fees - ship`, `true_profit = net - COGS`, `margin = profit/GMV*100`.

Fees estimated when blank: Shopify `2.9%+30c`, Etsy `6.5%+20c`, Amazon `15%`.

In [1]:
briefing = build_seller_briefing(events)
k = briefing['kpis']
print(f"GMV ${k['gmv']:.2f}  Fees ${k['fees']:.2f}  Ship ${k['shipping']:.2f}  Net ${k['net']:.2f}")
print(f"COGS ${k['cogs']:.2f}  True Profit ${k['true_profit']:.2f}  Margin {k['margin_pct']:.1f}%")
print(f"Orders {k['orders']}  AOV ${k['aov']:.2f}  Best: {k['best_seller']['product']} ({k['best_seller']['qty']})")

GMV $274.00  Fees $16.71  Ship $24.00  Net $233.29
COGS $64.20  True Profit $169.09  Margin 61.7%
Orders 8  AOV $34.25  Best: Tote Bag Handmade (6)


### KPI table (demo golden)

| metric | value |
|---|---|
| GMV | $274.00 |
| fees | $16.71 |
| shipping | $24.00 |
| net | $233.29 |
| COGS | $64.20 |
| **true_profit** | **$169.09** |
| margin | 61.7% |
| AOV | $34.25 |

Golden contract: profit `169.09 ±0.05` — evals/run.py asserts it.

In [1]:
print(f"Worst: {k['worst_seller']['product']} ({k['worst_seller']['qty']})  At-risk {k['at_risk']}  Stockout {k['stockout_risk']}")

Worst: Knitted Scarf (1)  At-risk 3  Stockout risk 1


## 4. Reorder math — velocity vs on-hand

`daily_velocity = sold_7d / 7`, `days_left = on_hand / velocity`, `reorder_qty = ceil(velocity*14 - on_hand)` (14d cover).

In [1]:
r = briefing['stockout_risk'][0]
print(r)
vel = r['sold_7d']/7
print(f"vel={vel:.2f}/d days_left={r['days_supply']} need=ceil({vel:.2f}*14 - {r['on_hand']}) = 10")

{'product': 'Tote Bag Handmade', 'on_hand': 2, 'sold_7d': 6, 'days_supply': 2.3, 'citation': '[inventory.csv:2] Tote Bag Handmade on_hand=2'}
vel=0.86/d days_left=2.3 need=ceil(0.86*14 - 2) = 10


### Reorder sheet (close)

Urgency: `<3d ORDER TODAY`, `<5d this week`, else `watch`. Tote = ORDER TODAY.

In [1]:
from omni_one.packs.seller_close import build_daily_close
from omni_one.packs.seller_scenarios import run_all_scenarios
scen = run_all_scenarios(events)
close = build_daily_close(events, briefing, scen, shop='demo')
print(close['reorder_sheet'])

[{'product': 'Tote Bag Handmade', 'on_hand': 2, 'sold_7d': 6, 'daily_velocity': 0.86, 'days_left': 2.3, 'reorder_qty': 10, 'cover_days': 14, 'supplier': 'Acme Textiles', 'urgency': 'ORDER TODAY', 'citation': '[inventory.csv:2] Tote Bag Handmade on_hand=2'}]


## 5. Reply drafts — who refunds if you sleep?

At-risk = negative DM/review (`hasn't arrived`, `no tracking`, `chipped`, `refund`). Money at risk ≈ `len(queue)*AOV`.

In [1]:
print(f"at-risk preview: {len(briefing['at_risk_preview'])} money_at_risk=${close['kpis']['money_at_risk_usd']:.2f}")
for q in close['reply_queue']:
    print(f"- [{q['priority']}] {q['customer']}: {q['summary'][:60]} | {q['citation']}")

at-risk preview: 3 money_at_risk=$102.75
- [P0-reply-in-2h] Alice: Alice: Hi! My tote bag hasn't arrived yet, ordered 5 days | [instagram_dm.txt:1] Alice: Hi! My tote bag hasn't arrived yet, ordered 5 days
- [P0-reply-in-2h] Charlie: Charlie: Where is my order? No tracking. | [instagram_dm.txt:4] Charlie: Where is my order? No tracking.
- [P0-reply-in-2h] customer: 2024-09-11 Ceramic Mug 2 Arrived chipped, very disappoi | [reviews.csv:3] 2024-09-11 Ceramic Mug 2 Arrived chipped, very disappointed


### Copy-paste draft (DM)

Template in `seller_close.draft_dm_reply`: apologize → cite order → promise 2h fix (reship/refund choice).

In [1]:
print(close['reply_queue'][0]['draft'])

Hi Alice! I'm really sorry about your late delivery — I just pulled your order ([instagram_dm.txt:1] Alice: Hi! My tote bag hasn't arrived yet, ordered 5 days) and I'm checking tracking/supplier right now. I'll update you within 2 hours with a fix (reship/refund — your choice). Thank you for your patience with demo!


### Review draft

`draft_review_reply`: thank → cite → fixed packing → offer replacement/refund.

In [1]:
print(close['reply_queue'][2]['draft'])

Thank you for flagging this ([reviews.csv:3] 2024-09-11 Ceramic Mug 2 Arrived chipped, very disappointed) — we're sorry about your customer experience. We've fixed the packing/quality check that caused it. Please DM us for a free replacement or refund — we want to make this right. — demo


## 6. Money leaks — dollar amounts, not charts

- `fee_creep` if fees/GMV ≥12%
- `shipping_loss` per order if ship/GMV ≥20%
- `margin_dip` if margin <10%

Demo has **0 leaks** — healthy margin 61.7%.

In [1]:
print(f"leaks: {close['money_leaks']}")
print(f"listing_fixes: {len(close['listing_fixes'])}")

leaks: []
listing_fixes: 3


### Listing fixes (tomorrow's sales)

Demo: REVIEW_CRISIS (Ceramic Mug) + 2× PRICE_MISMATCH (Etsy line-total vs Shopify unit quirk — see notebook 02). Worst seller = Knitted Scarf ×1 → bundle it.

In [1]:
for f in close['listing_fixes']:
    print(f"- {f['fix']}: {f['title'][:90]} | {f['evidence'][:60]}")

- REVIEW_CRISIS: Bad review — fix + public reply: 2024-09-11 Ceramic Mug 2 Arrived chipped, very disappoi | [reviews.csv:3] 2024-09-11 Ceramic Mug 2 Arrived chipped
- PRICE_MISMATCH: Price mismatch: Tote Bag Handmade {'etsy': 14.0, 'shopify': 28.0} (100% diff) | [orders] Tote Bag Handmade {'etsy': 14.0, 'shopify': 28.
- PRICE_MISMATCH: Price mismatch: Ceramic Mug {'etsy': 11.0, 'shopify': 22.0} (100% diff) — a | [orders] Ceramic Mug {'etsy': 11.0, 'shopify': 22.0}


## 7. Exports — hand accountant something

`write_close_bundle`: `close.json` + `close.csv` + `reorder.csv` + `replies.txt` + `po.txt`. Deterministic, idempotent.

In [1]:
from omni_one.packs.seller_close import close_to_csv, reorder_to_csv, replies_to_txt
print(close_to_csv(close)[:400])

DAILY CLOSE 2026-..,Profit $169.09 (61.7%) · $102.75 at risk in 3 angry messages · 1 reorder(s) TODAY · 0 leak(s) to plug
metric,value
gmv,274.0
fees,16.71
shipping,24.0
net,233.29
cogs,64.2
true_profit,169.09
margin_pct,61.7
orders,8
aov,34.25


### PO draft

`draft_po_email`: supplier + lines (product, qty, on-hand, days_left, citation) → confirm price + ETA.

In [1]:
print(close['po_draft'])

Subject: Reorder request — Acme Textiles

Hi Acme Textiles!

Please send the following for demo:
  - Tote Bag Handmade: 10 units (have 2, 2.3d left) [[inventory.csv:2] Tote Bag Handmade on_hand=2]

Please confirm price + ETA. Last invoice numbers are cited above from our records.

Thanks!
demo


## 8. Headline — one line over coffee

`Profit $X (Y%) · $Z at risk in N messages · M reorder(s) TODAY · L leak(s)`

In [1]:
print(close['headline'])
print(close['do_today'])

Profit $169.09 (61.7%) · $102.75 at risk in 3 angry messages · 1 reorder(s) TODAY · 0 leak(s) to plug
['Reply to 3 message(s) — $102.75 at risk (drafts below, cite file:line)', 'Reorder Tote Bag Handmade x10 from Acme Textiles (2.3d left)', 'No money leaks — push best seller']


## 9. Idempotency + checklist

Same folder → same headline + reorder (minus timestamp). Eval `run.py` reruns close to prove it.

- [ ] Reply 3 (P0 2h)
- [ ] Reorder Tote ×10 today
- [ ] Public reply to chipped-mug review
- [ ] Push best seller (no leaks)
- [ ] File close.csv/replies.txt

In [1]:
from omni_one.packs.seller_close import build_daily_close
c2 = build_daily_close(events, briefing, scen, shop='demo')
print(f"idempotent headline match: {c2['headline']==close['headline']}")
print(f"reorder match: {c2['reorder_sheet']==close['reorder_sheet']}")

idempotent headline match: True
reorder match: True
